In [2]:
# run to test if kernel is set up
import afqmctools

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# some global settings for convenience/consistency
U = 4.0
nelec = (16,16)
lattice_dims = (8,4)
lattice_params = {
    'L1' : lattice_dims[0],
    'L2' : lattice_dims[1],
    'boundary1' : 'PBC',
    'boundary2' : 'PBC'
}

In [ ]:
# set up the lattice and visualize
from afqmctools.systems.lattice import get_lattice
import afqmctools.utils.visualize as vis


lattice = get_lattice(
    params=lattice_params
)

vis.plot_lattice(lattice)

In [ ]:
from afqmctools.hamiltonian.model.director import HamiltonianDirector
import afqmctools.utils.io as io

# define Hamiltonian parameters
hamiltonian_params = { 
    'hamiltonian' : { 
        "U" : U  # note: nearest-neighbor hoping with t=1 is used by default
    } 
}

# ... and build it
hamiltonian = HamiltonianDirector(
    lattice=lattice,
    source=hamiltonian_params
).build()

# AND save it!
io.write_model_hamiltion(
    hamiltonian=hamiltonian,
    fname="afqmc.h5",
    nelec=nelec
)

In [ ]:
# get a trial wavefunction: First, let's try a free-electron (i.e. non-interacting) wavefunction
from afqmctools.wavefunction.model import write_free_electron_wfn

write_free_electron_wfn(
    hamiltonian_fname="afqmc.h5",
    nelec=nelec,
    U=U # for energy evaluation only!
)

## Make an input file with AFQMC run parameters

TODO: try breaking this down section-by-section!

A sample runfile is provided in the working directory of this tutorial: `afqmc.json`. 
The sample input file is reproduced below. 
Each section of the input file will be explained in more detail below.

```json
{
  "afqmc": {
    "project": {
      "id": "qmc",
      "series": 0,
      "mixed_precision": false
    },
    "execute": {
      "walker_set": {
        "walker_type": "COLLINEAR"
      },
      "wavefunction": {
        "filename": "afqmc.h5"
      },
      "hamiltonian": {
        "filename": "afqmc.h5"
      },
      "timestep": 0.01,
      "steps": 10000,
      "population_control_interval" : 10,
      "measure_interval_multiplier": 1,
      "walker_ortho_interval" : 10 ,
      "n_walkers_per_mpi_task": 10,
      "estimator": {
        "name": "energy",
        "overwrite": true,
        "print_components": true
      },
      "estimator": {
        "name":"back_propagation",
        "path_restoration":true,
        "extra_path_restoration":true,
        "bp_walker_ortho_interval":10,
        "measure_interval_multiplier":40,
        "naverages":1,
        "equil":200,
        "onerdm" : {
          "name":"onerdm"
	      }
      }
    }
  }
}
```

The outer block is an "afqmc" block which tells the code that we want to run an AFQMC calculaton.
The afqmc block consists of a few sub-blocks.

### The "project" block

The "project" block contains a few miscellaneous settings. A typical use case does not require these settings to be changed!

```json
{
  "afqmc": {
    "project": {
      "id": "qmc",
      "series": 0,
      "mixed_precision": false
    },
    ... 
  }
}
```


### The "execute" block

The "execute" block contains most of the settings that a typical user may want to change.
It contains a few parameters and a few sub-blocks.
Starting with the subblocks:

1. "walker_set" defines the settings used for the set of random walkers. Specifically, the type of walker is set here. Options include:  
  - "CLOSED" or RHF-like  
  - "COLINEAR" or UHF-like   
  - "NONCOLLINEAR" or GHF-like  
  - "FULLYPOLARIZED" - used when the down spin-sector has no electrons  

2. the "wavefunction" and the "hamiltonian" blocks give the name of the HDF5 file where the trial wavefunction and the Hamiltonian are saved, respectively. Note: if the wavefunction and the Hamiltonian reside in the same file, then only only the wavefunction block needs to be set!

```json
{
  "afqmc": {
    ...
    "execute": {
      "walker_set": {
        "walker_type": "COLLINEAR"
      },
      "wavefunction": {
        "filename": "afqmc.h5"
      },
      "hamiltonian": {
        "filename": "afqmc.h5"
      },

    ...

    }
  }
}
```

Next, the execute block contains the AFQMC run parameters.
All intervals are expressed in units of steps (i.e. population control, etc.).

```json
{
  "afqmc": {
    "project": {
      "id": "qmc",
      "series": 0,
      "mixed_precision": false
    },
    "execute": {
      
      ...
      
      "timestep": 0.01,          # imagninary time step size
      "steps": 10000,            # how many total steps to perform (total time = steps*timestep)
      "measure_interval_multiplier": 10,    # how often to measure (units of steps)
      "population_control_interval" : 10,         # how often to perform population control (units of steps)
      "walker_ortho_interval" : 10 ,             # how often to perform mGS orthogonalization of walkers (units of steps)
      "n_walkers_per_mpi_task": 10,           # number of walkers PER PROCESS
      
      ...
    
    }
  }
}
```

And finally, the execute block contains one or more "estimator" blocks.
this will almost always include at least one "energy" measurement block, see below.

```json
{
    ... 

    "execute": {
      
      ...
      
      "estimator": {
        "name": "energy",
        "overwrite": true,
        "print_components": true
      },

      ...
    
    }
  }
}
```

IF we want to compute observables that do not commute with the Hamiltonian - for example, the electron charge and/or spin density - then we need to include a back propagation (B.P.) estimator as well.
The B.P. estimator requires a few additional parameters to be set.
All time intervals are in units of steps.

```json
{
    ... 

    "execute": {
      
      ...
      
      "estimator": {
        "name":"back_propagation",
        "path_restoration":true,
        "extra_path_restoration":true,
        "bp_walker_ortho_interval":10,                      # How often to perform orthogonalization during B.P.   
        "measure_interval_multiplier":40,                    # How many step to use for B.P.   
        "equil":200,                    # number of equilibration steps before starting B.P.   
        "onerdm" : {                     # defines the observable that we want to measure    
          "name":"onerdm"
	      }
      }
    }
  }
}
```



#### multiple back propagation averages

The AFQMC code can compute multiple "averages" during B.P. as a means of saving time when checking for convergence in the measurement interval size.
If `naverages` is greater than 1, then B.P. is performed for each measurement interval in the range from `nsteps` / `naverages` to `nsteps` and the result is saved for each interval.
For example, if `nsteps` is 400 and `naverages` is 4, then B.P. is performed for each of 100, 200, 300, 400 steps.
This avoids re-running the forward propagation many times to test for convergence in B.P.

## Time to run the AFQMC executable

We can either submit a Slurm job or run the AFQMC executable locally.
The latter option is reasonable when the kernel used to run this notebook is hosted on a Flatiron workstation.

In [ ]:
# run AFQMC on rusty
!sbatch --wait run_afqmc.sh

In [ ]:
# OR run it locally - works reasonably well on workstations
!$AFQMC --filenames afqmc_local.json

## Analyze the AFQMC output

The AFQMC code saves "scalar" data, such at the energy, in a file called `qmc.s000.scalar.dat` (a text file)
and non-scalar data, such as the one-rdm and other obseervables, in a file called `qmc.s000.stat.h5` (HDF5 file).
The AFQMC code inlcudes tools to analyze both.


### Analyzing the AFQMC energy output

We will begin with analyzing the AFQMC energy output in `qmc.s000.scalar.dat` using the `scalar_stats` tool.
For more details on using this tool, run `$ scalar_stats --help`. At a minimum, we want to set the equilibration time using `-e [equlibration time in units of imaginary time (NOT steps!)]`. We can all set `-x time -t` to plot the AFQMC energy vs projection time, along with other useful information. When running remotely, i.e. on the cluster, the `--savefig [filename to save].png` option can be used to save the figure instead of plotting. 

In [ ]:
# analyze the energy - let's import this all here
!scalar_stats qmc.s000.scalar.dat -x time -t -e 10.0 --savefig energy_fe.png

Running the above outputs the energy, along with the standard error, followed by an estimate of the autocorrelation time.
At the end of the output line, the equilibration time and the total projection time are displayed as "equil/total".
The plot generated by the above is shown below.

![image](./energy_fe.png)

We see that AFQMC energy is equilibrated at our specified time of 10.0 $t^{-1}$. In general, we may need to rerun with a new equilibration time based on the curve.

### Analyzing the AFQMC one-rdm outpput

The one-rdm can be analyzed using the `afqmctools` python package.
Specifically, the `average_afqmc_rdm` function from the `afqmctools.analysis.rdm` module automates most of this.

A few features of `average_afqmc_rdm` are explained here for reference. 

1. `average_afqmc_rdm()` will return the average one-rdm and the asociated stochastic errorbars. The shape of the returned one-rdm (and stochastic error matrix) is (number of "averges") x (number of independent spin sectors) x Nsites x Nsites. The meaning of the number of averages in explained above in the Section [multiple back propagation averages](#multiple-back-propagation-averages).

2. setting equilibraiton time, We can set Teq (in units of t^{-1} NOT steps!) to discard samples from the beginning of the measurement. Recall that we also set an equilibration length in the B.P. settings of the input file. The AFQMC code does not record the one-rdm for times less than the equilibration length. Therefore, the *effective* equilbration time becomes the sum of the equilibration length 
set in the afqmc.json input file and Neq set here!  
```python
rho_avg, delta_rho = average_afqmc_rdm(Teq=1.2)
```  

***alternatively*** we can set $Neq$ to specify the equilbration time in measurement blocks. i.e. how many samples to throw out from the beginning of the calculation.

```python
rho_avg, delta_rho = average_afqmc_rdm(Neq=3)
```  


3. `average_afqmc_rdm` attempts to determine the autocorrelation length, $\kappa$, automatically; however, an autocorrelation length can also be specified as below. This can be used to test that automatically detected $\kappa$ value is correct - i.e. by repeating the averaging with different values of $\kappa$

```python
rho_avg, delta_rho = average_afqmc_rdm(kappa=3.0)
```  


Once we have an averaged one-rdm, we can use it compute various properties.
In this case, we will compute the charge-density and $\langle \hat{S}_z \rangle$.


In [ ]:
from afqmctools.analysis.rdm import average_afqmc_rdm


rho_avg, delta_rho = average_afqmc_rdm()

In [ ]:
# plot charge density

rho_total = (rho_avg[0,0].diagonal() + (rho_avg[0,1]).diagonal() ).reshape(lattice_dims)
delta_rho_total = np.sqrt(np.power(delta_rho[0,0],2) + np.power(delta_rho[0,1],2) ).diagonal().reshape(lattice_dims)

print(f"Integrated total number of electrons = {np.sum(rho_total)}")

fig,axs = plt.subplots(1,2,figsize=(12,8))

ax1 = axs[0]
ax1.set_title("charge density")
im = ax1.imshow(rho_total.real,cmap='seismic')
fig.colorbar(im,ax=ax1,cmap='seismic')


ax2 = axs[1]
im2 = ax2.imshow(delta_rho_total.real,cmap='seismic')
fig.colorbar(im2,ax=ax2,cmap='seismic')

ax2.set_title("delta charge density")


plt.show()

In [ ]:
# plot spin density
rho_spin_total = (rho_avg[0,0].diagonal() - (rho_avg[0,1]).diagonal() ).reshape(lattice_dims)
delta_rho_spin_total = np.sqrt(np.power(delta_rho[0,0],2) + np.power(delta_rho[0,1],2) ).diagonal().reshape(lattice_dims)

fig,axs = plt.subplots(1,2,figsize=(12,8))

print(rho_spin_total)

cmap = 'seismic'

ax1 = axs[0]
ax1.set_title("spin density")

im = ax1.imshow(rho_spin_total.real,cmap=cmap)
fig.colorbar(im,ax=ax1,cmap=cmap)


ax2 = axs[1]
im2 = ax2.imshow(delta_rho_spin_total.real,cmap=cmap)
fig.colorbar(im2,ax=ax2,cmap=cmap)

ax2.set_title("delta spin density")

plt.show()


### Improving the Trial Wavefunction

In the calculations above, we used a free electron trial wavefunction; however, this is not always the most accurate trial wavefunction especially for large U.
We can also use a trial wavefunction from Hartree-Fock (HF).
A lattice model HF solver is included with the AFQMC code which can be used to compute these trial wavefunctions.
Currently, both UHF and GHF are supported.


In [ ]:
# A better trial wavefunction: run HF
from cli.lattice_hartree_fock import lattice_hf

settings = {
    'nelec' : nelec,
    'plot' : True,
    'verbose' : True
}

# running this will automatically save the resulting wavefunction
#   in a file called "autoHF_wfn.h5" which can be directly used in the AFQMC code!
lattice_hf(hamiltonian,lattice,settings)

We need to update our input file to use this trial wavefunction instead. The file below is provided in the working directory of this tutorial as `afqmc_w_hf.json`.

The only difference from before is the "wavefunction" block which now specifies the HF trial wavefunction file.

```json
{
  "afqmc": {
    "project": {
      "id": "qmc",
      "series": 0,
      "mixed_precision": false
    },
    "execute": {
      "walker_set": {
        "walker_type": "COLLINEAR"
      },
      "wavefunction": {
        "filename": "autoHF_wfn.h5"
      },
      "hamiltonian": {
        "filename": "afqmc.h5"
      },
      "timestep": 0.01,
      "steps": 10000,
      "population" : 10,
      "measure_interval_multiplier": 1,
      "walker_ortho_interval" : 10 ,
      "n_walkers_per_mpi_task": 10,
      "estimator": {
        "name": "energy",
        "overwrite": true,
        "print_components": true
      },
      "estimator": {
        "name":"back_propagation",
        "path_restoration":true,
        "extra_path_restoration":true,
        "bp_walker_ortho_interval":10,
        "measure_interval_multiplier":40,
        "equil":200,
        "onerdm" : {
          "name":"onerdm"
	      }
      }
    }
  }
}
```

In [ ]:
# Now rerun AFQMC!
#. ... on rusty
!sbatch --wait run_afqmc_hf_trial.sh

In [ ]:
# ... or locally
!$AFQMC --filenames afqmc_local.json

### Analyze the new AFQMC results as before

first the energy...

In [ ]:
# analyze the energy - let's import this all here
!scalar_stats qmc.s000.scalar.dat -x time -t -e 10.0 --savefig energy_hf.png

![image](./energy_hf.png)

and now the one-rdm...

In [ ]:
rho_avg, delta_rho = average_afqmc_rdm()

In [ ]:
# plot charge density

rho_total = (rho_avg[0,0].diagonal() + (rho_avg[0,1]).diagonal() ).reshape(lattice_dims)
delta_rho_total = np.sqrt(np.power(delta_rho[0,0],2) + np.power(delta_rho[0,1],2) ).diagonal().reshape(lattice_dims)

print(f"Integrated total number of electrons = {np.sum(rho_total)}")

fig,axs = plt.subplots(1,2,figsize=(12,8))

ax1 = axs[0]
ax1.set_title("charge density")
im = ax1.imshow(rho_total.real,cmap='seismic')
fig.colorbar(im,ax=ax1,cmap='seismic')


ax2 = axs[1]
im2 = ax2.imshow(delta_rho_total.real,cmap='seismic')
fig.colorbar(im2,ax=ax2,cmap='seismic')

ax2.set_title("delta charge density")

plt.show()

In [ ]:
# plot spin density
rho_spin_total = (rho_avg[0,0].diagonal() - (rho_avg[0,1]).diagonal() ).reshape(lattice_dims)
delta_rho_spin_total = np.sqrt(np.power(delta_rho[0,0],2) + np.power(delta_rho[0,1],2) ).diagonal().reshape(lattice_dims)

fig,axs = plt.subplots(1,2,figsize=(12,8))

print(rho_spin_total)

cmap = 'seismic'

ax1 = axs[0]
ax1.set_title("spin density")

im = ax1.imshow(rho_spin_total.real,cmap=cmap)
fig.colorbar(im,ax=ax1,cmap=cmap)


ax2 = axs[1]
im2 = ax2.imshow(delta_rho_spin_total.real,cmap=cmap)
fig.colorbar(im2,ax=ax2,cmap=cmap)

ax2.set_title("delta spin density")

plt.show()